# Reproducible Fokker–Planck credit-scoring analysis

This notebook runs from the original UCI credit-card spreadsheet through classifier benchmarking, calibration-only threshold selection, the eight-value PDE coupling sweep, grid/diffusion sensitivity, numerical diagnostics, held-out predictions, tables, and figures. All primary test metrics use the same untouched test split. PDE step size: 0.1; maximum steps: 200. German Credit external validation is a separate experiment, not executed here.

## Data loading, partitions and evaluation metrics

In [3]:
import os,time,json,platform
from pathlib import Path
import numpy as np,pandas as pd
import matplotlib;matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,balanced_accuracy_score,roc_auc_score,average_precision_score,roc_curve,precision_recall_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier,HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from scipy.ndimage import gaussian_filter1d
from scipy.sparse import diags,eye
from scipy.sparse.linalg import splu
from scipy.stats import spearmanr
from xgboost import XGBClassifier
from itertools import product
root=Path.cwd()/'results';root.mkdir(exist_ok=True)
raw=pd.read_excel(str(Path.cwd()/'default of credit card clients.xlsx'),sheet_name='Data',header=1)
raw.columns=[str(c).strip() for c in raw.columns]
y=(1-raw['default payment next month']).to_numpy(int);g=(raw.SEX.to_numpy()==2).astype(int)
features=[c for c in raw if c not in ['ID','SEX','default payment next month']]
X=raw[features].to_numpy(float);idx=np.arange(len(y));strata=y*2+g
train,rest=train_test_split(idx,test_size=.4,random_state=42,stratify=strata)
cal,test=train_test_split(rest,test_size=.5,random_state=42,stratify=strata[rest]);yc=y[cal];yt=y[test];gc=g[cal];gt=g[test]
assert len(set(train)&set(cal))==len(set(train)&set(test))==len(set(cal)&set(test))==0
pd.DataFrame({'index':idx,'partition':np.where(np.isin(idx,train),'train',np.where(np.isin(idx,cal),'calibration','test'))}).to_csv(root/'partitions_reconciled.csv',index=False)
thresholds=np.linspace(.05,.95,901)
def choose_tau(z):
 b=np.array([balanced_accuracy_score(yc,z>=t) for t in thresholds]);return float(thresholds[np.argmax(b)])
def calc(name,pred,score,lam=np.nan):
 ap=[pred[gt==k].mean() for k in (0,1)];tp=[pred[(gt==k)&(yt==1)].mean() for k in (0,1)];fp=[pred[(gt==k)&(yt==0)].mean() for k in (0,1)]
 return dict(Intervention=name,lambda_=lam,Accuracy=accuracy_score(yt,pred),Balanced_accuracy=balanced_accuracy_score(yt,pred),ROC_AUC=roc_auc_score(yt,score),PR_AUC=average_precision_score(yt,score),DP_gap=abs(ap[0]-ap[1]),EO_gap=abs(tp[0]-tp[1]),FPR_gap=abs(fp[0]-fp[1]),Approval_0=ap[0],Approval_1=ap[1],TPR_0=tp[0],TPR_1=tp[1],FPR_0=fp[0],FPR_1=fp[1])


## Fit classifiers and select calibration thresholds

In [4]:
models={
'Logistic Regression':make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000,random_state=42)),
'Random Forest':RandomForestClassifier(n_estimators=350,min_samples_leaf=2,random_state=42,n_jobs=4),
'HistGradientBoosting':HistGradientBoostingClassifier(max_iter=350,learning_rate=.045,max_leaf_nodes=15,random_state=42),
'XGBoost':XGBClassifier(n_estimators=350,max_depth=3,learning_rate=.045,subsample=.85,colsample_bytree=.9,objective='binary:logistic',eval_metric='logloss',tree_method='hist',random_state=42,n_jobs=4)}
bench=[];scores={};taus={}
for name,m in models.items():
 m.fit(X[train],y[train]);c=m.predict_proba(X[cal])[:,1];s=m.predict_proba(X[test])[:,1];t=choose_tau(c)
 bench.append(dict(Classifier=name,threshold=t,**calc(name,s>=t,s)));scores[name]=(c,s);taus[name]=t
 print('MODEL',name,t,bench[-1]['Accuracy'],flush=True)
pd.DataFrame(bench).to_csv(root/'table1_classifier_benchmarks.csv',index=False)
model=models['XGBoost'];sc,st=scores['XGBoost'];tau=taus['XGBoost'];baseline=calc('XGBoost, no transport',st>=tau,st)
base_bacc=balanced_accuracy_score(yc,sc>=tau)
opts=[]
for t0,t1 in product(np.linspace(.45,.95,51),repeat=2):
 p=np.where(gc==0,sc>=t0,sc>=t1);ba=balanced_accuracy_score(yc,p)
 if ba>=base_bacc-.03:opts.append((abs(p[gc==0].mean()-p[gc==1].mean()),-ba,float(t0),float(t1)))
dpcal,_,t0,t1=min(opts);comparison=calc('Group-specific threshold',np.where(gt==0,st>=t0,st>=t1),st)
print('thresholds',tau,t0,t1,flush=True)


MODEL Logistic Regression 0.7449999999999999 0.757
MODEL Random Forest 0.7739999999999999 0.7321666666666666
MODEL HistGradientBoosting 0.7819999999999999 0.761
MODEL XGBoost 0.7629999999999999 0.7703333333333333
thresholds 0.7629999999999999 0.64 0.69


## Conservative PDE implementation

In [5]:
# Independent conservative finite-volume solver: frozen nonlocal drift, implicit Euler dt=0.1
lams=[0,.0025,.005,.01,.02,.05,.1,.2];dt=.1;tol=1e-8;K=200

def prepare(N,D):
 h=1/N;edges=np.linspace(0,1,N+1)
 def density(z):
  counts,_=np.histogram(z,bins=edges);r=gaussian_filter1d(counts.astype(float),sigma=2,mode='reflect')+1e-6;return r/(h*r.sum())
 rho0=np.stack([density(sc[gc==k]) for k in (0,1)])
 drift=np.stack([D*np.diff(r)/(h*np.where(np.diff(r)>=0,r[:-1],r[1:])) for r in rho0])
 fraction=np.clip((edges[1:]-np.maximum(edges[:-1],tau))/h,0,1)
 def gap(r):return float(h*np.dot(fraction,r[0]-r[1]))
 def transport(z,old,new):
  oldcdf=np.r_[0,np.cumsum(old)*h];newcdf=np.r_[0,np.cumsum(new)*h];u=np.interp(z,edges,oldcdf);uq,ind=np.unique(newcdf,return_index=True);return np.interp(u,uq,edges[ind])
 return rho0,drift,gap,transport,h

def solve(lam,N=400,D=.001,record=False):
 rho0,drift,gap,transport,h=prepare(N,D);rho=rho0.copy();diag=[];start=time.perf_counter();converged=(lam==0)
 if lam:
  for j in range(K):
   delta=gap(rho);nxt=np.empty_like(rho)
   for k in (0,1):
    v=drift[k]-lam*(1 if k==0 else -1)*delta
    a=np.maximum(v,0)+D/h;c=np.maximum(-v,0)+D/h
    main=np.zeros(N);main[:-1]-=a/h;main[1:]-=c/h
    L=diags([a/h,main,c/h],[-1,0,1],shape=(N,N),format='csc')
    nxt[k]=splu(eye(N,format='csc')-dt*L).solve(rho[k])
   change=float(h*np.abs(nxt-rho).sum(axis=1).max());rho=nxt
   diag.append(dict(step=j+1,time=(j+1)*dt,delta=gap(rho),energy=.5*gap(rho)**2,L1_change=change,min_density=float(rho.min()),mass_error=float(np.max(abs(rho.sum(axis=1)*h-1)))))
   if change<tol:converged=True;break
 else:diag=[dict(step=0,time=0,delta=gap(rho),energy=.5*gap(rho)**2,L1_change=0,min_density=float(rho.min()),mass_error=float(np.max(abs(rho.sum(axis=1)*h-1))))]
 z=st.copy()
 for k in (0,1):z[gt==k]=transport(st[gt==k],rho0[k],rho[k])
 m=calc('PDE '+str(lam),z>=tau,z,lam)
 m.update(initial_density_gap=gap(rho0),terminal_density_gap=gap(rho),steps=len(diag) if lam else 0,converged=converged,max_mass_error=max(q['mass_error'] for q in diag),min_density=min(q['min_density'] for q in diag),runtime=time.perf_counter()-start,N=N,D=D)
 if record:pd.DataFrame(diag).to_csv(root/f'pde_diagnostics_lambda_{lam:g}_reconciled.csv',index=False)
 return m,z,rho,rho0,pd.DataFrame(diag)
rows=[];zs={};rhos={};diagnostics={}
for lam in lams:
 m,z,rho,rho0,diag=solve(lam,record=True);rows.append(m);zs[lam]=z;rhos[lam]=rho;diagnostics[lam]=diag
 print('PDE',lam,m['Accuracy'],m['DP_gap'],m['terminal_density_gap'],flush=True)
pde=pd.DataFrame(rows);pde.to_csv(root/'table4_coupling_sweep.csv',index=False)
comparison_df=pd.DataFrame([baseline,comparison]+[rows[lams.index(l)] for l in [.02,.05,.1,.2]])
comparison_df.to_csv(root/'table3_comparison.csv',index=False)
pd.DataFrame(bench)[['Classifier','Approval_0','Approval_1','TPR_0','TPR_1','FPR_0','FPR_1']].to_csv(root/'table2_group_rates.csv',index=False)
pde[['lambda_','initial_density_gap','terminal_density_gap','DP_gap','steps','converged']].to_csv(root/'table5_density_vs_test.csv',index=False)


PDE 0 0.7703333333333333 0.0506625012364863 -0.05672894221608925
PDE 0.0025 0.7706666666666667 0.04537846697989445 -0.05145976874252242
PDE 0.005 0.7718333333333334 0.041474885511816995 -0.046820847616609866
PDE 0.01 0.7713333333333333 0.033943813133365475 -0.03912043783757946
PDE 0.02 0.7708333333333334 0.024059585404399564 -0.02835543239083274
PDE 0.05 0.7696666666666667 0.011703313876186061 -0.014117326310723185
PDE 0.1 0.7691666666666667 0.004749965053297833 -0.007425146782374797
PDE 0.2 0.769 0.0016512026541846136 -0.0037857217859601023


## Full coupling sweep and primary tables

In [6]:
# 3x3 diffusion/grid sensitivity, lambda 0.2
sens=[]
for D in [.0005,.001,.002]:
 for N in [200,400,800]:
  m,*_=solve(.2,N=N,D=D);sens.append(m);print('SENS',D,N,flush=True)
sens=pd.DataFrame(sens);sens.to_csv(root/'table6_grid_diffusion.csv',index=False)
pde[['lambda_','steps','max_mass_error','min_density','runtime','converged']].to_csv(root/'table7_solver_diagnostics.csv',index=False)
# Monotonicity from heldout score pairs, actual ties possible
mono=[]
for lam,z in zs.items():
 for k in (0,1):
  ix=np.argsort(st[gt==k],kind='stable');s=st[gt==k][ix];t=z[gt==k][ix]
  violations=int(np.sum(np.diff(t)<-1e-12));corr=float(spearmanr(s,t).statistic)
  mono.append(dict(lambda_=lam,Group=k,Violations=violations,Spearman=corr))
pd.DataFrame(mono).to_csv(root/'table8_monotonicity.csv',index=False)
pd.DataFrame({'index':test,'group':gt,'outcome':yt,'baseline_score':st,'threshold_comparator_decision':np.where(gt==0,st>=t0,st>=t1),**{f'pde_{lam:g}':z for lam,z in zs.items()}}).to_csv(root/'heldout_scores_all_methods.csv',index=False)


SENS 0.0005 200
SENS 0.0005 400
SENS 0.0005 800
SENS 0.001 200
SENS 0.001 400
SENS 0.001 800
SENS 0.002 200
SENS 0.002 400
SENS 0.002 800


## Sensitivity and monotonicity checks

In [7]:
# Figures
plt.figure(figsize=(7.5,4.2));plt.plot(pde.lambda_,100*np.abs(pde.terminal_density_gap),'o-',label='Calibration density gap');plt.plot(pde.lambda_,100*pde.DP_gap,'s--',label='Held-out decision gap');plt.xlabel('Fairness coupling strength $\\lambda$');plt.ylabel('Absolute DP gap (percentage points)');plt.grid(alpha=.25);plt.legend();plt.tight_layout();plt.savefig(root/'Fig22_terminal_gap_vs_lambda.pdf');plt.close()
plt.figure(figsize=(7,4));
for lam in [.02,.1,.2]:
 d=diagnostics[lam];plt.plot(d.time,d.energy,label=f'$\\lambda={lam}$')
plt.yscale('log');plt.xlabel('Dimensionless PDE time');plt.ylabel('Fairness energy');plt.legend();plt.tight_layout();plt.savefig(root/'Fig21_fairness_energy_decay.png',dpi=170);plt.close()
fig,axs=plt.subplots(1,3,figsize=(12,3.5));sgrid=np.linspace(.55,.95,81);threshold_data=[]
for t in sgrid:
 for name,z in [('Baseline',st),('PDE lambda=0.20',zs[.2])]:
  m=calc(name,z>=t,z);threshold_data.append(dict(threshold=t,method=name,accuracy=m['Accuracy'],DP=m['DP_gap'],EO=m['EO_gap']))
threshold_df=pd.DataFrame(threshold_data);threshold_df.to_csv(root/'threshold_sensitivity_reconciled.csv',index=False)
for name,grp in threshold_df.groupby('method'):
 for ax,metric in zip(axs,['accuracy','DP','EO']):ax.plot(grp.threshold,grp[metric],label=name)
for ax,title in zip(axs,['Accuracy','Absolute DP gap','Absolute EO gap']):ax.set_title(title);ax.set_xlabel('Decision threshold');ax.grid(alpha=.2)
axs[0].legend(fontsize=8);fig.tight_layout();fig.savefig(root/'threshold_sensitivity.png',dpi=170);plt.close(fig)
fig,axs=plt.subplots(1,2,figsize=(9,3.7))
for name,z in [('Baseline',st),('PDE lambda=0.20',zs[.2])]:
 fpr,tpr,_=roc_curve(yt,z);pr,rc,_=precision_recall_curve(yt,z)
 axs[0].plot(fpr,tpr,label=f'{name} (AUC {roc_auc_score(yt,z):.3f})');axs[1].plot(rc,pr,label=f'{name} (AP {average_precision_score(yt,z):.3f})')
axs[0].set(xlabel='False positive rate',ylabel='True positive rate');axs[1].set(xlabel='Recall',ylabel='Precision')
for ax in axs:ax.legend(fontsize=8);ax.grid(alpha=.2)
fig.tight_layout();fig.savefig(root/'roc_pr_curves.png',dpi=170);plt.close(fig)
fig,ax=plt.subplots(figsize=(7,4))
for D,grp in sens.groupby('D'):ax.plot(grp.N,grp.runtime,'o-',label=f'D={D:g}')
ax.set(xlabel='Number of spatial cells',ylabel='Runtime (s)');ax.legend();fig.tight_layout();fig.savefig(root/'solver_runtime_scaling.png',dpi=170);plt.close(fig)
for name,col,ylabel in [('Fig17_l1_density_convergence.png','L1_change','Successive density $L^1$ change'),('Fig18_positivity_diagnostic.png','min_density','Minimum density'),('Fig19_mass_conservation_diagnostic.png','mass_error','Absolute mass error')]:
 fig,ax=plt.subplots(figsize=(7,4))
 for lam in [.02,.1,.2]:ax.plot(diagnostics[lam].step,diagnostics[lam][col],label=f'$\\lambda={lam}$')
 ax.set(xlabel='Iteration',ylabel=ylabel);ax.legend();fig.tight_layout();fig.savefig(root/name,dpi=170);plt.close(fig)
# terminal vs reference stationary, only comparison to initial density (not claimed exact equilibrium)
fig,axs=plt.subplots(1,2,figsize=(9,3.4));s=(np.arange(400)+.5)/400
for k,ax in enumerate(axs):ax.plot(s,rho0[k],label='Initial density');ax.plot(s,rhos[.2][k],label='Finite-horizon terminal');ax.set_title(f'Group {k}');ax.set_xlabel('Creditworthiness score');ax.legend(fontsize=8)
fig.tight_layout();fig.savefig(root/'Fig20_terminal_vs_stationary_density.png',dpi=170);plt.close(fig)


## Generate manuscript figures

In [8]:
# create metadata
settings=dict(seed=42,train=len(train),calibration=len(cal),test=len(test),xgboost=models['XGBoost'].get_params(),tau=tau,group_thresholds=[t0,t1],dt=dt,spatial_cells=400,diffusivity=.001,max_steps=K,convergence_tolerance=tol,lambda_grid=lams,threshold_objective='minimize calibration DP gap subject to balanced accuracy >= common-threshold calibration balanced accuracy - 0.03')
(root/'experiment_settings_reconciled.json').write_text(json.dumps(settings,indent=2,default=str))
print('DONE',root,flush=True)


DONE /Users/olusolaolabanjo/Desktop/pde-credit-scoring/run-init/results
